# CerberusAI - Entrenamiento QLoRA en Google Colab

Re-entrena `Qwen/Qwen2.5-Coder-7B-Instruct` con **QLoRA 4-bit** sobre los datasets
`v2 + v3 + v4_balance` (1594 ejemplos) y deja el adapter LoRA en Google Drive.

Requisitos: **Colab Pro** con GPU (L4 24 GB / A100 40 GB). El merge a GGUF y la
cuantizacion se hacen **en local** (ver la ultima celda). Hiperparametros identicos
a la corrida de referencia (5 epochs, batch 2, grad-accum 4, seq 2048, seed 42).


## 0. Verificar la GPU


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("bf16 soportado:", torch.cuda.is_bf16_supported())


## 1. Montar Google Drive

El adapter se guarda en `MyDrive/CerberusAI/` para que sobreviva al fin de la sesion.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Clonar el repositorio (publico)


In [ ]:
REPO = "https://github.com/Coliiin1/CerberusIDE-IA.git"
BRANCH = "01-apiexpuesta"
!git clone -b {BRANCH} {REPO} /content/CerberusIDE-IA
%cd /content/CerberusIDE-IA


## 3. Instalar dependencias

Se usa el `torch` CUDA que ya trae Colab (no se reinstala).


In [ ]:
!pip install -q -r CerberusAI/finetune/requirements-colab.txt


## 4. Construir los datos (v2 + v3 + v4)

Concatena los cinco `.jsonl`, los convierte a formato chat de Qwen y hace el split
estratificado (seed 42). Esperado: 1594 ejemplos -> train ~1275 / eval ~319.


In [ ]:
%%bash
cd /content/CerberusIDE-IA/CerberusAI
python finetune/to_chat.py --input \
    dataset/cerberus_dataset_v2.jsonl \
    dataset/cerberus_dataset_v3_valido.jsonl \
    dataset/cerberus_dataset_v3_errores_semanticos.jsonl \
    dataset/cerberus_dataset_v3_errores_sintacticos.jsonl \
    dataset/cerberus_dataset_v4_balance.jsonl \
    --output finetune/chat_all.jsonl
python finetune/split.py --input finetune/chat_all.jsonl --out-dir finetune --eval-ratio 0.2 --seed 42
python finetune/train_qlora.py --check


## 5. Entrenar

`--dtype auto` usa `bf16` en L4/A100 (y `fp16` si tocara una T4).
Si hay OOM: anade `--batch 1 --grad-accum 8`.


In [ ]:
%cd /content/CerberusIDE-IA/CerberusAI
import os
OUT = "/content/drive/MyDrive/CerberusAI/lora-adapter"
os.makedirs(OUT, exist_ok=True)
!python finetune/train_qlora.py --output "{OUT}" --epochs 5 --dtype auto
# Si hay OOM en L4:  --batch 1 --grad-accum 8


## 6. Empaquetar y descargar el adapter


In [ ]:
!cd /content/drive/MyDrive/CerberusAI && zip -r lora-adapter.zip lora-adapter
from google.colab import files
files.download("/content/drive/MyDrive/CerberusAI/lora-adapter.zip")


## 7. Merge / GGUF / Ollama (en local)

1. Descarga y descomprime `lora-adapter.zip` en `CerberusAI/finetune/lora-adapter/`.
2. `py CerberusAI\finetune\export_gguf.py` (CPU, ~28 GB RAM).
3. Sigue las instrucciones que imprime: `convert_hf_to_gguf.py` + `llama-quantize Q8_0`.
4. `ollama create cerberusai -f CerberusAI\finetune\Modelfile`.
5. Evalua: `py CerberusAI\eval\run_eval.py`.
